# Session 8 — Tabular Q-learning, an RL-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Practice session (about 80 minutes)**

I am Salih Tutun. I talk. You type when I say so. Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Neighbor, two voices, then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

**Family this session: RL-based agent.** The Q-table fills the decide box that an LLM-based agent fills with a tool call.

![Same loop, different decide](../../assets/desk-q-decide.png)

![Q-learning update](../../assets/q-update.png)

## Run of show
- 10 min — The Q-table is the "decide" box from Session 1, filled by learning instead of an LLM.
- 55 min — Finish the update and the ε-greedy choice. Read the diagnostics.
- 15 min — What would make you refuse to deploy this table?

## Environment
Same supermarket Home-and-lifestyle shelf as Session 7. Demand is sampled from the real invoices. State = stock 0–4. Action = order 0, 1, or 2. You do **not** get `P(demand)` in the update. You only see the next stock and the reward after you order.

## The update, before you code it

**ε-greedy** is the same rule as Session 4, now over a row of the Q-table. With probability `epsilon`, pick an action in {0, 1, 2} at random. Otherwise pick the action with the largest Q in that row. Ties break toward the smaller index.

**Q-learning** is the temporal-difference update from Session 7, written for one state and one action. After you take `action` in `stock`, receive `pay`, and land in `ending`:

- `target = pay + GAMMA * max(q_table[ending])`
- `q_table[stock, action] = q_table[stock, action] + alpha * (target - q_table[stock, action])`

`target` is the one-step backup: the reward you just got, plus a discounted guess of the best action from the next stock. The term in parentheses is the surprise. `alpha` is how hard that surprise moves the table. You never use `P(demand)`. The `max` is what makes it Q-learning rather than a method that follows the action you happened to take next.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I explain. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor. Two people share. Then type `YOU TYPE HERE`. |
| **TODO** / `YOU WRITE THIS` | You write the function. Search for `TODO`. We stay until every line says PASS. |
| **RUN WITH ME** | Already written. Run it. Do not type in that cell. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** — and one line for your project framework. |


## Your work — TODO (you write this in class)

Search for `TODO`. Complete `epsilon_greedy` and the Q update. Run the cell once **before** you edit it so you see the empty-update return.


In [ ]:
import numpy as np

from pathlib import Path
from collections import Counter, defaultdict
import csv

def course_data(name):
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(name)


def empirical_demand_p():
    """Bin daily Home-and-lifestyle quantity into {0, 1, 2}."""
    daily = defaultdict(int)
    with course_data("supermarket-sales.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product line") or "").strip() != "Home and lifestyle":
                continue
            daily[row["Date"]] += int(float(row["Quantity"]))
    totals = sorted(daily.values())
    q1, q2 = totals[len(totals) // 3], totals[(2 * len(totals)) // 3]
    bins = [0 if qty <= q1 else 1 if qty <= q2 else 2 for qty in totals]
    counts = Counter(bins)
    n = max(len(bins), 1)
    return {level: counts[level] / n for level in (0, 1, 2)}


DEMAND_P = empirical_demand_p()
DEMAND_LEVELS = list(DEMAND_P.keys())
DEMAND_PROBS = [DEMAND_P[k] for k in DEMAND_LEVELS]
print("Empirical P(demand) from Home-and-lifestyle invoices:", {k: round(v, 3) for k, v in DEMAND_P.items()})

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5
RNG = np.random.default_rng(4)


def step(stock, order):
    demand = int(RNG.choice(DEMAND_LEVELS, p=DEMAND_PROBS))
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    missed = max(0, demand - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay


def epsilon_greedy(q_row, epsilon, rng):
    """TODO: with probability epsilon return a random action in {0, 1, 2}; else the best action in q_row."""
    # TODO
    return 0


def q_learning(episodes=400, alpha=0.1, epsilon=0.15):
    q_table = np.zeros((CAPACITY + 1, 3))
    recent = []
    for episode in range(episodes):
        stock = int(RNG.integers(0, CAPACITY + 1))
        total = 0.0
        for _ in range(15):
            action = epsilon_greedy(q_table[stock], epsilon, RNG)
            ending, pay = step(stock, action)
            # TODO: TD target and Q update
            # target = pay + GAMMA * max(q_table[ending])
            # q_table[stock, action] += alpha * (target - q_table[stock, action])
            total += pay
            stock = ending
        recent.append(total)
    return q_table, np.mean(recent[-50:])


q_table, last_return = q_learning()
print(np.round(q_table, 2))
print("mean return of last 50 episodes", round(last_return, 3))
print("A learned table orders more when stock is 0 than when stock is 4.")


## Diagnostics

After your update works, change one knob at a time and rerun:

| Change | What you should watch |
| --- | --- |
| `epsilon=0` | The table can freeze on the first action that got lucky. |
| `alpha=1` | The table tracks the last sample and oscillates. |
| Reward with `STOCKOUT = 0` | The policy may stop caring about empty shelves. That is a misspecified reward, not a coding win. |

## Debrief
Would you put this table on a real SKU tomorrow? Name the human override.

## Fill this with me — do this in class (no empty box)

**STOP. Do not scroll.** Neighbor 20s. Two voices. Then we type the `TODO` together.

1. Run the learning cell once **before** you edit the `TODO`. Say the mean return.
2. Write the Q update together. Run it again. The last-50-episode return should climb.
3. Change one knob from the table above and tell the room what broke.


## Add this to your framework — STOP. Do not scroll.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep.

Hint: the Q-table **is** the decide box from Session 1, filled by learning. A cracked-mug email is not a Q-row — that stays LLM plus a reviewer.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
